# เฉลย Lab 02 — ทำความรู้จักกับ Titanic

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df = sns.load_dataset("titanic")[["survived", "pclass", "sex", "age",
                                  "sibsp", "parch", "fare", "embarked"]]

### ข้อ 1 — สำรวจ

891 แถว × 8 คอลัมน์ แต่ละแถว = ผู้โดยสาร 1 คน · missing มากสุด: age 177 ค่า (19.9%)

In [2]:
print(df.shape)
df.info()

(891, 8)
<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   survived  891 non-null    int64  
 1   pclass    891 non-null    int64  
 2   sex       891 non-null    str    
 3   age       714 non-null    float64
 4   sibsp     891 non-null    int64  
 5   parch     891 non-null    int64  
 6   fare      891 non-null    float64
 7   embarked  889 non-null    str    
dtypes: float64(2), int64(4), str(2)
memory usage: 55.8 KB


In [3]:
df.describe()

,survived,pclass,age,sibsp,parch,fare
count,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


สิ่งผิดปกติที่เจอ (ตอบได้หลายอย่าง):
- `fare` min = 0 → ใครนั่งฟรี? (ต้องตามดู)
- `fare` max 512 vs median 14.45 → เบ้มาก mean หลอกตา
- `age` min 0.42 → ทารก 4 เดือน เป็นไปได้จริง

In [4]:
df["embarked"].value_counts(dropna=False)

embarked
S      644
C      168
Q       77
NaN      2
Name: count, dtype: int64

### ข้อ 2 — missing

เหตุผลของเรา: age เบ้ → ใช้ **median** (28) · embarked หายแค่ 2 แถว → ใช้ **most_frequent** (S)
และเก็บ `age_missing` เป็นฟีเจอร์เพิ่ม เพราะ missing สัมพันธ์กับชั้นโดยสาร (ชั้น 3 หายมากกว่า) = มีสัญญาณ

In [5]:
df["age_missing"] = df["age"].isna().astype(int)
print(df.groupby("pclass")["age_missing"].mean().round(3))   # ยืนยันว่า missing มีแบบแผน

df["age"] = df["age"].fillna(df["age"].median())
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])
df.isna().sum()

pclass
1    0.139
2    0.060
3    0.277
Name: age_missing, dtype: float64


survived       0
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
embarked       0
age_missing    0
dtype: int64

### ข้อ 3 — target

base rate 38.4% · โมเดลที่ทายทุกคนว่าตายได้ accuracy = 61.6% (61.6% ของผู้โดยสารเสียชีวิตจริง)

In [6]:
print("survival rate:", round(df["survived"].mean(), 3))
print(df.groupby("sex")["survived"].mean().round(3))
print(df.groupby("pclass")["survived"].mean().round(3))

survival rate: 0.384
sex
female    0.742
male      0.189
Name: survived, dtype: float64
pclass
1    0.630
2    0.473
3    0.242
Name: survived, dtype: float64


ผู้หญิง 74.2% vs ผู้ชาย 18.9% · ชั้น 1: 63% / ชั้น 2: 47% / ชั้น 3: 24%

### ข้อ 4 — duplicates & outliers

duplicate 111 แถว (ผู้โดยสารคนละคนที่ค่า 8 คอลัมน์ซ้ำกัน) → เรา drop เพื่อกันโอกาส duplicate
ข้าม split แล้วทำคะแนนบวม · outlier fare 116 แถวตาม IQR แต่ fare>300 คือตั๋วจริงของครอบครัวรวย (รอดสูง) → **เก็บไว้**

In [7]:
print("duplicates:", df.duplicated().sum())
df = df.drop_duplicates()

q1, q3 = df["fare"].quantile([0.25, 0.75])
fence = q3 + 1.5 * (q3 - q1)
print("outliers (IQR):", (df["fare"] > fence).sum())
df[df["fare"] > 300][["pclass", "sex", "fare", "survived"]]

duplicates: 111
outliers (IQR): 97


,pclass,sex,fare,survived
258,1,female,512.3292,1
679,1,male,512.3292,1
737,1,male,512.3292,1


### ข้อ 5 — ผู้หญิงชั้น 3 vs ผู้ชายชั้น 1

In [8]:
pd.pivot_table(df, index="pclass", columns="sex", values="survived").round(3)

sex,female,male
pclass,,
1,0.968,0.378
2,0.917,0.185
3,0.472,0.159


เฉลย: **ผู้หญิงชั้น 3 (~50%) รอดมากกว่าผู้ชายชั้น 1 (~37%)** — เพศดันสำคัญกว่าชั้น!

"women and children first" ถูกบังคับใช้จริงในปี 1912 — แต่ถ้าเกิดซ้ำวันนี้กฎอาจต่างไป
ประเด็นสำคัญทาง ML: โมเดลเรียนรู้จากอดีต ถ้าโลกเปลี่ยน โมเดลก็ต้องเรียนใหม่